# Part 2: Job Postings Analysis

Generative AI - Assignment 1
Darrsheni Sapovadia (27PGAI0063)

This dataset is a scrape of job boards, with a job title and a long description for each
posting. The brief asks for the first 25. For every posting I work out a broad category for
the role, then pull the required skills, education level and experience out of the
description.

Same setup as Part 1: LangChain on top of Groq, using `openai/gpt-oss-120b`.

## Setup

In [1]:
import json
import os
import re
import time

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

load_dotenv("../.env")

MODEL = os.getenv("LLM_MODEL", "openai/gpt-oss-120b")

# Same settings as Part 1. reasoning_effort is turned down because this model
# counts its own thinking against max_tokens, and on a long job description it
# will happily use the entire budget reasoning and return nothing at all.
llm = ChatGroq(model=MODEL, temperature=0, max_tokens=800, reasoning_effort="low")

print("Model:", MODEL)

Model: openai/gpt-oss-120b


In [2]:
def ask(chain, values):
    """Run a chain, waiting it out if Groq rate limits us.

    The free tier allows 8,000 tokens a minute and some of these descriptions run
    to several thousand characters, so being limited is normal here. Groq puts
    the wait it wants in the error message, so I use that rather than guessing.
    """
    for attempt in range(6):
        try:
            return chain.invoke(values)
        except Exception as error:
            message = str(error)
            limited = "rate" in message.lower() or "429" in message
            if not limited or attempt == 5:
                raise
            suggested = re.search(r"try again in ([\d.]+)s", message)
            time.sleep(float(suggested.group(1)) + 2 if suggested else 20)


def find_json(text):
    """Grab the JSON object out of a reply and ignore anything around it."""
    found = re.search(r"\{.*\}", text, re.S)
    if not found:
        return {}
    try:
        return json.loads(found.group())
    except json.JSONDecodeError:
        return {}

## Step 1: Load the dataset

The CSV has an unnamed index column left over from however it was saved, so I drop that and
rename the other two to something tidier.

In [3]:
jobs = pd.read_csv("../data/job_title_des.csv")

print("Whole dataset:", jobs.shape)
print("Columns:", list(jobs.columns))
jobs.head(3)

Whole dataset: (2277, 3)
Columns: ['Unnamed: 0', 'Job Title', 'Job Description']


,Unnamed: 0,Job Title,Job Description
0,0,Flutter Developer,We are looking for hire experts flutter develo...
1,1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."


In [4]:
df = jobs.head(25).copy().reset_index(drop=True)
df = df.drop(columns=["Unnamed: 0"])
df = df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"})

print("Working set:", df.shape)
print()
print("Description length: shortest", df["Job_Description"].str.len().min(),
      "chars, longest", df["Job_Description"].str.len().max())
df.head()

Working set: (25, 2)

Description length: shortest 405 chars, longest 7645


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore, IN\n..."
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outside o...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2: Job category classification

I kept the category list from the brief. Worth saying up front that this particular dataset
is scraped almost entirely from IT job boards, so nearly everything here is going to come
back as Technology/IT. The list still earns its place because it gives the model somewhere
to put a posting that is not a developer role, and `Others` is there as the fallback the
brief asks for.

In [5]:
DOMAINS = ["Technology/IT", "Finance", "Marketing", "Healthcare", "Education", "Others"]

category_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You sort job postings into one broad domain. "
     "Reply with the domain name only, nothing else."),
    ("human",
     "Given the following job title and description, categorize the job into one of the "
     "following domains: Technology/IT, Finance, Marketing, Healthcare, Education, Others.\n\n"
     "Job: Staff Accountant\n"
     "Description: Prepare monthly reconciliations, assist with year end audit and "
     "maintain the general ledger.\n"
     "Domain category:"),
    ("ai", "Finance"),
    ("human",
     "Job: Backend Engineer\n"
     "Description: Build REST APIs in Python, work with PostgreSQL and deploy on AWS.\n"
     "Domain category:"),
    ("ai", "Technology/IT"),
    ("human",
     "Job: {title}\n"
     "Description: {description}\n"
     "Domain category:"),
])

category_chain = category_prompt | llm | StrOutputParser()

In [6]:
def classify_job(title, description):
    reply = ask(category_chain, {"title": title, "description": description[:1500]})
    for domain in DOMAINS:
        if domain.lower() in reply.lower():
            return domain
    # "IT" on its own is a common reply and would miss the check above
    if "tech" in reply.lower() or " it" in reply.lower():
        return "Technology/IT"
    return "Others"

In [7]:
sample = df.loc[0]

print("Title:", sample["Job_Title"])
print("Predicted category:", classify_job(sample["Job_Title"], sample["Job_Description"]))

Title: Flutter Developer


Predicted category: Technology/IT


## Step 3: Requirements extraction

The brief allows either three separate prompts or one combined prompt that returns all
three fields. I went with the combined one. It is a third of the API calls, and it means the
model reads the description once and fills in all three fields from the same pass rather
than three disconnected reads.

Plenty of these postings simply do not mention a degree or a number of years, so I tell the
model to write `Not specified` rather than let it invent something.

In [8]:
requirements_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You read job descriptions and pull out the requirements. "
     "Reply with JSON and nothing else. "
     "Never guess: if the description does not mention something, say \"Not specified\"."),
    ("human",
     "Extract the required skills, education level and years of experience from the job "
     "description below.\n\n"
     "Reply in this exact shape:\n"
     '{{"skills": ["..."], "education": "...", "experience": "..."}}\n\n'
     "skills: the tools, programming languages and specific abilities asked for.\n"
     "education: the minimum degree required or preferred, or \"Not specified\".\n"
     "experience: the years or level of experience asked for, or \"Not specified\".\n\n"
     "Job title: {title}\n"
     "Job description:\n{description}"),
])

requirements_chain = requirements_prompt | llm | StrOutputParser()

In [9]:
def extract_requirements(title, description):
    reply = ask(requirements_chain, {"title": title, "description": description[:2500]})
    data = find_json(reply)

    skills = data.get("skills", [])
    if isinstance(skills, str):
        skills = [skills]
    skills = [str(s).strip() for s in skills if str(s).strip()]

    education = str(data.get("education", "") or "").strip() or "Not specified"
    experience = str(data.get("experience", "") or "").strip() or "Not specified"

    return skills, education, experience

In [10]:
skills, education, experience = extract_requirements(
    sample["Job_Title"], sample["Job_Description"]
)

print("Title:", sample["Job_Title"])
print()
print("Skills:    ", skills)
print("Education: ", education)
print("Experience:", experience)

Title: Flutter Developer

Skills:     ['Flutter']
Education:  Not specified
Experience: 1 year (Preferred)


Worth a look at the raw description for that first posting, because it is a good example of
how thin some of these adverts are:

In [11]:
print(sample["Job_Description"])

We are looking for hire experts flutter developer. So you are eligible this post then apply your resume.
Job Types: Full-time, Part-time
Salary: ₹20,000.00 - ₹40,000.00 per month
Benefits:
Flexible schedule
Food allowance
Schedule:
Day shift
Supplemental Pay:
Joining bonus
Overtime pay
Experience:
total work: 1 year (Preferred)
Housing rent subsidy:
Yes
Industry:
Software Development
Work Remotely:
Temporarily due to COVID-19


## Step 4: Run the chains over all 25 postings

Two calls per posting, so fifty in total.

In [12]:
categories = []
all_skills = []
all_education = []
all_experience = []

start = time.time()

for position, row in df.iterrows():
    title = row["Job_Title"]
    description = row["Job_Description"]

    categories.append(classify_job(title, description))

    skills, education, experience = extract_requirements(title, description)
    all_skills.append(skills)
    all_education.append(education)
    all_experience.append(experience)

    print(f"posting {position + 1} of {len(df)} done")
    time.sleep(2)

print(f"\nAll finished in {time.time() - start:.0f} seconds")

posting 1 of 25 done


posting 2 of 25 done


posting 3 of 25 done


posting 4 of 25 done


posting 5 of 25 done


...

posting 22 of 25 done


posting 23 of 25 done


posting 24 of 25 done


posting 25 of 25 done



All finished in 214 seconds


## Step 5: Add the new columns

`Required_Skills` is kept as a comma separated string so the column is consistent and reads
properly when it gets written out to CSV.

In [13]:
df["Predicted_Category"] = categories
df["Required_Skills"] = [", ".join(s) if s else "Not specified" for s in all_skills]
df["Education_Required"] = all_education
df["Experience_Required"] = all_experience

# the four new columns on their own first
df[["Job_Title", "Predicted_Category", "Required_Skills",
    "Education_Required", "Experience_Required"]].head(10)

,Job_Title,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,Technology/IT,Flutter,Not specified,1 year (Preferred)
1,Django Developer,Technology/IT,"Python, Django, Flask, REST API development, R...",Not specified,Not specified
2,Machine Learning,Technology/IT,"Python, Java, Machine Learning, Deep Learning,...","Graduate or M.Sc. in Computer Science, Mathema...",At least 3 years
3,iOS Developer,Technology/IT,"Objective-C, Cocoa Touch, Core Data, Core Anim...",Not specified,Not specified
4,Full Stack Developer,Technology/IT,"React, React Native, JavaScript, HTML, CSS, RE...",Computer Science or equivalent,"5+ years web development, 2+ years React"
5,Java Developer,Technology/IT,"C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQL...",Bachelor's Degree in Computer Science,2 years
6,Full Stack Developer,Technology/IT,"Node.js, Java, NoSQL, MongoDB, Elasticsearch, ...","B.Sc degree in Computer Science, Engineering, ...",Minimum 2 years
7,JavaScript Developer,Technology/IT,"ReactJS, NodeJS, Azure Functions, GraphQL, HTM...",Any graduation,3-8 years
8,DevOps Engineer,Technology/IT,"Bash, Ruby, Python, Java, Puppet, Chef, Cloudi...",Not specified,Not specified
9,Software Engineer,Technology/IT,"REST API, C/C++, Linux/Unix, Python, Go, Git, ...",Not specified,Minimum 7 years


### The full dataframe, original columns and new ones together

In [14]:
pd.set_option("display.max_colwidth", 45)
df

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter d...,Technology/IT,Flutter,Not specified,1 year (Preferred)
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code...,Technology/IT,"Python, Django, Flask, REST API developme...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\n\nBangalore,...",Technology/IT,"Python, Java, Machine Learning, Deep Lear...","Graduate or M.Sc. in Computer Science, Ma...",At least 3 years
3,iOS Developer,JOB DESCRIPTION:\n\nStrong framework outs...,Technology/IT,"Objective-C, Cocoa Touch, Core Data, Core...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – ...,Technology/IT,"React, React Native, JavaScript, HTML, CS...",Computer Science or equivalent,"5+ years web development, 2+ years React"
5,Java Developer,Software Developer - Integration*\nImmedi...,Technology/IT,"C#, .NET, .NET Core, HTML5, CSS3, MsSQL, ...",Bachelor's Degree in Computer Science,2 years
6,Full Stack Developer,senior full stack developer \- 1800026h c...,Technology/IT,"Node.js, Java, NoSQL, MongoDB, Elasticsea...","B.Sc degree in Computer Science, Engineer...",Minimum 2 years
7,JavaScript Developer,"Job Description:\n\nReactJS + NodeJs, Azu...",Technology/IT,"ReactJS, NodeJS, Azure Functions, GraphQL...",Any graduation,3-8 years
8,DevOps Engineer,Main Responsibilities and Deliverables:\n...,Technology/IT,"Bash, Ruby, Python, Java, Puppet, Chef, C...",Not specified,Not specified
9,Software Engineer,"Overview\n\n\nBased in Silicon Valley, Ti...",Technology/IT,"REST API, C/C++, Linux/Unix, Python, Go, ...",Not specified,Minimum 7 years


### One row as JSON

Same shape as the example in the assignment brief.

In [15]:
row = df.loc[3]

print(json.dumps({
    "Job_Title": row["Job_Title"],
    "Job_Description": row["Job_Description"][:200] + "... [excerpt]",
    "Predicted_Category": row["Predicted_Category"],
    "Required_Skills": [s.strip() for s in row["Required_Skills"].split(",")],
    "Education_Required": row["Education_Required"],
    "Experience_Required": row["Experience_Required"],
}, indent=2))

{
  "Job_Title": "iOS Developer",
  "Job_Description": "JOB DESCRIPTION:\n\nStrong framework outside of iOS is always a plus\n\niOS experience and generalist engineers with backgrounds in related technologies is a plus\n\nA disciplined approach to development, d... [excerpt]",
  "Predicted_Category": "Technology/IT",
  "Required_Skills": [
    "Objective-C",
    "Cocoa Touch",
    "Core Data",
    "Core Animation",
    "Core Graphics",
    "Core Text",
    "third-party libraries",
    "APIs",
    "networking",
    "mobile network issues",
    "concurrency",
    "threading",
    "internationalization",
    "iOS frameworks",
    "mobile development lifecycle"
  ],
  "Education_Required": "Not specified",
  "Experience_Required": "Not specified"
}


### Spot check

The brief asks to verify a few of the outputs by hand. Rather than read all 25 I looked at
how often each field came back as `Not specified`, which points straight at the postings
worth reading properly.

In [16]:
print("Categories found:")
print(df["Predicted_Category"].value_counts().to_string())
print()

for column in ["Education_Required", "Experience_Required"]:
    missing = (df[column] == "Not specified").sum()
    print(f"{column}: {missing} of {len(df)} postings do not state this")

Categories found:
Predicted_Category
Technology/IT    25

Education_Required: 10 of 25 postings do not state this
Experience_Required: 7 of 25 postings do not state this


In [17]:
# reading three of them back against the description to check the extraction is sensible
for position in [1, 5, 9]:
    row = df.loc[position]
    print("-" * 70)
    print("Title:     ", row["Job_Title"])
    print("Category:  ", row["Predicted_Category"])
    print("Skills:    ", row["Required_Skills"][:160])
    print("Education: ", row["Education_Required"])
    print("Experience:", row["Experience_Required"])

----------------------------------------------------------------------
Title:      Django Developer
Category:   Technology/IT
Skills:     Python, Django, Flask, REST API development, RPC, Linux, SQL, JSON, Automated unit testing (PyUnit), Verbal and written communication
Education:  Not specified
Experience: Not specified
----------------------------------------------------------------------
Title:      Java Developer
Category:   Technology/IT
Skills:     C#, .NET, .NET Core, HTML5, CSS3, MsSQL, MySQL, ReactJS, Web services (WSDL, SOAP, RESTful), Relational databases, Data access/database queries, MVC architectur
Education:  Bachelor's Degree in Computer Science
Experience: 2 years
----------------------------------------------------------------------
Title:      Software Engineer
Category:   Technology/IT
Skills:     REST API, C/C++, Linux/Unix, Python, Go, Git, Gerrit, Jenkins, configuration and deployment of large systems, cloud platforms, VMs, containers, control or data 
Education

## Save the results

In [18]:
df.to_csv("../outputs/part2_job_results.csv", index=False)

print("Saved", len(df), "rows to outputs/part2_job_results.csv")

Saved 25 rows to outputs/part2_job_results.csv
